In [1]:
import os,json
from langchain_community. vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_community.document_loaders import json_loader

from dotenv import load_dotenv
load_dotenv()

import logfire


/var/folders/l3/s9nylmbn68g6rt9sqfv_26tw0000gn/T/ipykernel_30372/3539951559.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community. vectorstores import FAISS


In [2]:
knowledge_base=os.getcwd()+'/document.json'

In [3]:
# Load document (Knowledge base)

with open(knowledge_base) as f:
    data=json.load(f)

data

[{'topic': 'RAG',
  'source': 'doc1_',
  'content': "Retrieval-augmented generation (RAG) combines information retrieval with text generation. A system searches a collection of relevant documents for a user's query, then supplies the retrieved passages as context to a language model so the answer can be grounded in that material."},
 {'topic': 'Finetuning',
  'source': 'doc_2',
  'content': 'Fine-tuning adapts a pretrained language model by training it further on examples for a particular task, style, or domain. The training examples typically pair inputs with desired outputs, and optimization adjusts model parameters to make the model more likely to produce those outputs.'},
 {'topic': 'Evals',
  'source': 'doc_3',
  'content': 'LLM evaluations (evals) are repeatable tests that measure whether a model or application meets defined quality criteria. An evaluation set contains representative inputs and expected outcomes or grading criteria; running the system on that set produces measure

In [4]:
# converting raw data to langchain compatible document




DOCS=[Document(metadata={"topic":raw["topic"], "source":raw["source"]},page_content=raw["content"])
for raw in data ]

print(f" Loaded {len(DOCS)} documents:{[d.metadata["topic"] for d in DOCS]}")

 Loaded 6 documents:['RAG', 'Finetuning', 'Evals', 'Observability', 'Prompting', 'LLM Gateways']


In [5]:
DOCS

[Document(metadata={'topic': 'RAG', 'source': 'doc1_'}, page_content="Retrieval-augmented generation (RAG) combines information retrieval with text generation. A system searches a collection of relevant documents for a user's query, then supplies the retrieved passages as context to a language model so the answer can be grounded in that material."),
 Document(metadata={'topic': 'Finetuning', 'source': 'doc_2'}, page_content='Fine-tuning adapts a pretrained language model by training it further on examples for a particular task, style, or domain. The training examples typically pair inputs with desired outputs, and optimization adjusts model parameters to make the model more likely to produce those outputs.'),
 Document(metadata={'topic': 'Evals', 'source': 'doc_3'}, page_content='LLM evaluations (evals) are repeatable tests that measure whether a model or application meets defined quality criteria. An evaluation set contains representative inputs and expected outcomes or grading criter

In [6]:
# Cpnverting Docs to embedding and store to vector store



embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview",api_key=os.getenv("Gemini_API"))

vector_store=FAISS.from_documents(DOCS,embeddings)
retriever=vector_store.as_retriever(search_kwargs={"k":2})

print("FAISS index ready ✅")



FAISS index ready ✅


DATA INGESTION IS COMPELETED

#### DATA RETRIEVAL PIPELINE 

In [7]:
# Initiating logfire

logfire.configure(service_name="RAG_TRACE")
logfire.instrument_openai()

Logfire project URL: https://logfire-us.pydantic.dev/syednoormujassum/first-logfire

In [8]:
from langchain_groq import ChatGroq

llm_groq=ChatGroq(api_key=os.getenv("Groq_API"),model="openai/gpt-oss-20b")


In [34]:
def rag(question: str, user_id: str = "anonymous") -> str:
    with logfire.span("RAGPipeline", question=question, userid=user_id):
        docs = retriever.invoke(question)
        logfire.info(
            "Docs Retrieved",
            topics=[doc.metadata["topic"] for doc in docs],
            num_docs=len(docs),
        )

        context = "\n\n".join(
            f"[{doc.metadata['topic']}] {doc.page_content}"
            for doc in docs
        )

        prompt = (
            "Answer the question using only the context below.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\n"
            "Answer concisely. If the context does not contain the answer, say so."
        )
        response=llm_groq.invoke(prompt).content
        logfire.info("LLM_Output", response=response)
        return response

In [38]:
answer=rag("How do RAG reduce hallucination")


02:50:56.964 RAGPipeline
02:50:57.280   Docs Retrieved
02:50:57.281   Chat Completion with 'openai/gpt-oss-20b' [LLM]
02:50:57.858   LLM_Output


In [56]:
answer2=rag("explain fine tuning")

02:16:23.197 RAGPipeline
02:16:23.920   Docs Retrieved
02:16:26.866   LLM_Output


In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
logfire.instrument_openai()

gemini_llm=ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("Gemini_API"),
    temperature=0.5,
    model="gemini-3.5-flash-lite"

)

In [23]:
def rag2(question: str, user_id: str = "anonymous") -> str:
    with logfire.span("RAGPipeline", question=question, userid=user_id):
        docs = retriever.invoke(question)
        logfire.info(
            "Docs Retrieved",
            topics=[doc.metadata["topic"] for doc in docs],
            num_docs=len(docs),
        )

        context = "\n\n".join(
            f"[{doc.metadata['topic']}] {doc.page_content}"
            for doc in docs
        )

        prompt = (
            "Answer the question using only the context below.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\n"
            "Answer concisely. If the context does not contain the answer, say so."
        )
        response=gemini_llm.invoke(prompt).content
        logfire.info("LLM_Output", response=response)
        return response

In [24]:
answer=rag2("what is gateways")

02:29:58.427 RAGPipeline
02:29:58.813   Docs Retrieved
02:29:58.816   Chat Completion with 'gemini-3.5-flash-lite' [LLM]
02:30:03.848   LLM_Output


In [36]:
llm_groq = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("Groq_API"),
    model="openai/gpt-oss-20b",
    temperature=0.3,
)